In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import re 
import pandas as pd

2026-01-31 09:21:38.125994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769851298.343055      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769851298.405124      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769851298.887886      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769851298.887927      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769851298.887930      24 computation_placer.cc:177] computation placer alr

In [2]:
# import pandas as pd

# pd.set_option("display.max_colwidth", None)

In [3]:
train_df=pd.read_csv("/kaggle/input/llm-refusal-responses-dataset/train.csv")
test_df=pd.read_csv("/kaggle/input/llm-refusal-responses-dataset/test.csv")

In [4]:
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub("", text)

train_df["response"] = train_df["response"].astype(str).apply(remove_emojis)
test_df["response"] = test_df["response"].astype(str).apply(remove_emojis)


In [5]:
def clean_text(text):
    # Remove emojis
    text = remove_emojis(text)
    
    # Remove non-printable characters
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)
    
    # Remove excessive spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

train_df["response"] = train_df["response"].astype(str).apply(clean_text)
test_df["response"] = test_df["response"].astype(str).apply(clean_text)

In [6]:
# import re

# role_pattern = r"\b(USER|HUMAN|ASSISTANT|SYSTEM)\b"

# filtered_df = train_df[
#     train_df["response"].str.contains(role_pattern, regex=True, na=False)
# ]


In [7]:
# filtered_df

In [8]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

Train size: 35012, Val size: 3891


In [9]:
class RejectionDataset(Dataset):
    def __init__(self, texts, labels, embedding_model):
        self.texts = texts
        self.labels = labels
        self.embedding_model = embedding_model

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # SentenceTransformer expects a single string, not list
        embedding = self.embedding_model.encode(
            self.texts[idx],
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        x = torch.tensor(embedding, dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)

        return x, y


In [10]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

emb_model = SentenceTransformer(
    "intfloat/multilingual-e5-large-instruct",
    device=device
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [11]:
training_data=RejectionDataset(np.array(train_df["response"]),np.array(train_df["label"]),embedding_model=emb_model)
val_data=RejectionDataset(np.array(val_df["response"]),np.array(val_df["label"]),embedding_model=emb_model)
test_data=RejectionDataset(np.array(test_df["response"]),np.array(test_df["label"]),embedding_model=emb_model)

In [12]:
class RejectionClassifier(nn.Module):
    """
    FIX:
    - Output is 1 logit (not sigmoid)
    - BCEWithLogitsLoss will handle sigmoid
    """

    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.fc(x)


In [13]:
input_dim = emb_model.get_sentence_embedding_dimension()
classifier = RejectionClassifier(input_dim).to(device)

In [14]:
def overfit_one_batch(model, dataset, batch_size=32, epochs=300, lr=1e-2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    x, y = next(iter(loader))

    x = x.to(device)
    y = y.to(device).unsqueeze(1)  

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        preds = (torch.sigmoid(logits) >= 0.5).float()
        acc = (preds == y).float().mean().item()

        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss: {loss.item():.4f} | "
                f"Acc: {acc:.4f}"
            )


In [15]:
# overfit_one_batch(classifier,training_data)

In [16]:
def evaluate(model, val_dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in tqdm(val_dataloader):
            x = x.to(device)
            y = y.to(device).unsqueeze(1) 

            logits = model(x)             
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(f"\nVal Loss: {avg_loss:.4f} | Val Accuracy: {accuracy:.4f}")

    return avg_loss, accuracy


In [ ]:
def train(model, train_dataset, val_dataset,
          batch_size=64, epochs=20, learning_rate=1e-3, patience=3):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            y = y.to(device).unsqueeze(1)   

            optimizer.zero_grad()

            logits = model(x)               
            loss = criterion(logits, y)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            optimizer.step()

            total_loss += loss.item() * y.size(0)

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == y).sum().item()
            total_samples += y.size(0)

        train_loss = total_loss / total_samples
        train_acc = total_correct / total_samples

        val_loss, val_acc = evaluate(model, val_loader)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                model.load_state_dict(torch.load("best_model.pth"))
                break

    return model


In [18]:
classifier_model=train(classifier, training_data, val_data)

100%|██████████| 61/61 [02:53<00:00,  2.84s/it]



Val Loss: 0.2093 | Val Accuracy: 0.9190
Epoch 01 | Train Loss: 0.2438 | Train Acc: 0.9059 | Val Loss: 0.2093 | Val Acc: 0.9190


100%|██████████| 61/61 [02:53<00:00,  2.85s/it]



Val Loss: 0.2776 | Val Accuracy: 0.8772
Epoch 02 | Train Loss: 0.1912 | Train Acc: 0.9263 | Val Loss: 0.2776 | Val Acc: 0.8772


100%|██████████| 61/61 [02:53<00:00,  2.84s/it]


...


Val Loss: 0.1595 | Val Accuracy: 0.9427
Epoch 14 | Train Loss: 0.1069 | Train Acc: 0.9593 | Val Loss: 0.1595 | Val Acc: 0.9427


100%|██████████| 61/61 [02:54<00:00,  2.85s/it]



Val Loss: 0.1411 | Val Accuracy: 0.9496
Epoch 15 | Train Loss: 0.1026 | Train Acc: 0.9615 | Val Loss: 0.1411 | Val Acc: 0.9496


100%|██████████| 61/61 [02:53<00:00,  2.85s/it]


Val Loss: 0.1464 | Val Accuracy: 0.9504
Epoch 16 | Train Loss: 0.0981 | Train Acc: 0.9635 | Val Loss: 0.1464 | Val Acc: 0.9504
Early stopping triggered.
